## Devoir Maison “Econometric Evaluation of Public Policies”

Groupe 4 - François Bradesi, Fanny Daubet, Titouan Morvan, Raphaël Thireau

In [72]:
# Imports et lecture du jeu de données
import pandas as pd
import pyreadr
import statsmodels.formula.api as smf

result = pyreadr.read_r('df_group4.Rda')
df = result['df_group4']

### 1. Early stage analysis

Question 1: Given the description of the policy, build a variable Gi equal to 1 if municipality i belongs to a treated province and 0 otherwise. Run a descriptive statistic analysis to compare the groups {G = 0} and {G = 1}. Are these groups very different in terms of outcomes? If so, is it a problem? Are these groups very different in terms of the other variables? If so, is it a problem?

In [73]:
## Question 1

# Variable Gi valant 1 si la commune i appartient à une province traitée et 0 sinon
treated_provinces = [2, 3, 6]
df['G'] = df['province_index'].isin(treated_provinces).astype(int)

# Satistiques descriptives pour comparer les groupes {G = 0} et {G = 1}

### 2. Analysis without covariates

#### 2.1 Pseudo-parallel trend tests

#### 2.2 Average treatment effect estimation

### 3. Analysis with covariates
The two variables urban and organic_school_canteen can serve as covariates here.

Question 7: Write down the parallel trend assumptions required to identify and estimate the ATTs when one uses a DID approach with covariates. What is the other assumption needed to run a DID analysis with covariates? Verify this second assumption in the data.

In [74]:
# "When using a DID approach with covariates, the required assumption is the conditional parallel trends assumption.
# Formally, for each treatment group 𝑔 and period 𝑡:

# E[𝑌𝑡(0)−𝑌𝑡-1(0) ∣ 𝐺=1, 𝑋] = E[𝑌𝑡(0)−𝑌𝑡-1(0) ∣ 𝐺=0, 𝑋]

# where:
# 𝑌𝑡(0) is the potential outcome without treatment,
# 𝐺 indicates treatment status,
# 𝑋 are the covariates (here: urban and organic_school_canteen).

# The other key assumption is :

# S(P_{X|G=1}) ⊂ S(P_{X|G=0})

# where S(P) stands for the support of law P.

# Verification of the overlap assumption
urban_tab = pd.crosstab(df['G'], df['urban'])
organic_tab = pd.crosstab(df['G'], df['organic_school_canteen'])

print(urban_tab)
print(organic_tab)

# Les valeurs des variables urban et organic_school_canteen prises par les communes dans le groupe traité sont les mêmes que celles prises par celles du groupe traité.
# Les supports des lois conditionnelles sont donc identiques, ce qui valide l'hypothèse.

urban     0     1
G                
0      2058   681
1       940  1321
organic_school_canteen   0.0  1.0
G                                
0                       1907  832
1                       1453  808


Question 8: Let Xi stand for a vector stacking the variables urbani and organic_school_canteeni for
municipality i. A classmate of yours suggests to run the following regression to recover the ATTs
(using years 2017, 2018 and 2019):

ln_avg_household_organicit = α + βGi + γ1{t ≥ 2018} + δGi × 1{t ≥ 2018} + Xi'μ + εit. (1)

Do you think this regression is well-suited to identify both the ATTs in 2018 and 2019? (hint:
what may be the limitation of this regression if the ATTs at the two dates are not equal?) Which
coefficient of this regression is supposed to capture treatment effects? Estimate the ATTs with
this regression and build a 95% confidence interval (do not forget to cluster standard errors at
the municipality level).

In [75]:
# Cette spécification identifie uniquement un effet moyen post-traitement et suppose que :
# ATT2018​ = ATT2019
# La modèle proposé est donc mal spécifié si les effets diffèrent entre 2018 et 2019.
# Le coefficient d’intérêt est δ (interaction G × post).

In [76]:
# Création d'un identifiant de municipalité
df['municipality_id'] = df.index

# Transformation des données en panel
df_long = df.melt(
    id_vars=['municipality_id', 'province_index', 'urban', 'organic_school_canteen', 'G'],
    value_vars=[
        'ln_avg_household_organic_2016',
        'ln_avg_household_organic_2017',
        'ln_avg_household_organic_2018',
        'ln_avg_household_organic_2019'
    ],
    var_name='year',
    value_name='ln_avg_household_organic'
)

# extraire l'année
df_long['year'] = df_long['year'].str[-4:].astype(int)

# variable post
df_long['post'] = (df_long['year'] >= 2018).astype(int)

# Modèle
model = smf.ols(
    'ln_avg_household_organic ~ G + post + G:post + urban + organic_school_canteen',
    data=df_long
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_long['municipality_id']}
)

print(model.summary())

                               OLS Regression Results                               
Dep. Variable:     ln_avg_household_organic   R-squared:                       0.616
Model:                                  OLS   Adj. R-squared:                  0.616
Method:                       Least Squares   F-statistic:                 1.597e+04
Date:                    mar., 03 mars 2026   Prob (F-statistic):               0.00
Time:                              19:56:41   Log-Likelihood:                -10207.
No. Observations:                     20000   AIC:                         2.043e+04
Df Residuals:                         19994   BIC:                         2.047e+04
Df Model:                                 5                                         
Covariance Type:                    cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------

In the next two questions, you are expected to estimate separately the two ATTs using the OR approach introduced in class in order to obtain alternative treatment effect estimates to what was obtained with Model (1). As a reminder, the OR approach requires estimating a mt(·) function (see the slides for definitions).

Question 9: Estimate the mt(·) function using a linear regression approach. Build a 95% confidence interval for each ATT (you may have to resort to the bootstrap). Comment on the results and compare with what you obtained without covariates.

Question 10: Estimate the mt(·) function using a fully nonparametric approach (hint: you only have two binary covariates so this is very straightforward if you look carefully at the slides). Build a 95% confidence interval for each ATT (you may have to resort to the bootstrap). Comment on the results and compare with what you obtained without covariates.

Question 11: For this last question, focus on years 2017 and 2018. The goal is to detect potential heterogeneity of treatment effects in year 2018 (still using a DID approach with years 2017 and 2018). Give the definition of the CATT function in the present context. Propose a treatment effect heterogeneity test based on the CATT function (several possibilities are proposed in the slides, choose the one you prefer). Implement this test and comment on the result.